# TinyTroupe Demo: Gen Z Skincare Concept Test

This notebook runs a small **virtual concept test** for a Gen Z (US, ages 18–28) skincare product, using [TinyTroupe](https://github.com/microsoft/TinyTroupe) — Microsoft's framework for multi-agent LLM-driven persona simulation.

## What it does

1. **Generates** 10 Gen Z US personas from `usa.json` demographics via `TinyPersonFactory`.
2. **Runs** a one-question concept test (three skincare concepts, one preferred choice + explanation per persona).
3. **Extracts** structured results (preferred option, reasoning, purchase likelihood, improvement) and saves them to `result.csv`.
4. **Compares** the result against a parallel run using a different engine (see [claude-persona genz-skincare demo](https://github.com/takechanman1228/claude-persona/tree/main/demo/genz-skincare)).

## Concepts under test

- **A — Acne Control Serum**: fights breakouts with clinically proven actives
- **B — Barrier Repair Cream**: strengthens skin barrier and reduces redness
- **C — Glow Boosting Toner**: brightens dull skin and supports a more even complexion

Default model: `gpt-5-mini` (the TinyTroupe 0.7.0 default; configured in `config.ini`).


In [ ]:
import os
# Set OPENAI_API_KEY in your shell BEFORE launching Jupyter (e.g. `export OPENAI_API_KEY=sk-...`).
assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your shell before running this notebook."


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsExtractor
import tinytroupe.control as control


In [ ]:
control.begin('cache.json')


## 1. Generate the persona panel

Synthesize 10 Gen Z (ages 18–28) US personas from `usa.json` demographics using TinyTroupe's `TinyPersonFactory` (the Microsoft factory idiom). First run takes ~1–2 minutes; reruns hit the simulation cache (`cache.json`) and are near-instant.


In [ ]:
TinyPerson.clear_agents()

factory = TinyPersonFactory.create_factory_from_demography(
    "./usa.json",
    population_size=10,
    additional_demographic_specification="Gen Z skincare shoppers in the US (ages 18-28).",
)
panel = factory.generate_people(10, temperature=1.3)

print(f"✅ Generated {len(panel)} personas")
for a in panel:
    occ = a.get("occupation")
    occ_title = occ.get("title") if isinstance(occ, dict) else occ
    print(f"  - {a.get('name')} | age {a.get('age')} | {occ_title}")


## 2. Run the concept test

Each persona reacts to all three concepts in an isolated `TinyWorld` and is asked to pick one, justify the choice, rate purchase likelihood, and suggest one improvement.


In [ ]:
%%time
CONCEPTS = """
**Option A: Acne Control Serum** — fights breakouts with clinically proven actives.

**Option B: Barrier Repair Cream** — strengthens skin barrier and reduces redness.

**Option C: Glow Boosting Toner** — brightens dull skin and supports a more even complexion.
"""

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

for i, agent in enumerate(panel, start=1):
    print(f"[{i}/{len(panel)}] Interviewing: {agent.get('name')}...")
    world = TinyWorld(f"ConceptTest_Room_{run_ts}_{i}", [agent])
    world.broadcast("Welcome. I'd like your reaction to three new skincare product concepts we're developing.")
    world.broadcast(
        f"Here are the three concepts:\n{CONCEPTS}\n\n"
        "Which ONE (A, B, or C) is your top choice? Please explain why, how likely you'd actually be to buy it, "
        "and what one thing you'd change about it to make it better."
    )
    world.run(1)
    print(f"    ✓ done")


In [ ]:
# ResultsExtractor uses the LLM to pull structured fields out of each persona's free-text response.
extractor = ResultsExtractor()

def extract_concept_results(panel, situation):
    rows = []
    for agent in panel:
        extraction = extractor.extract_results_from_agent(
            agent,
            extraction_objective="Extract the persona's concept-test choice, reasoning, purchase likelihood and improvement suggestion.",
            situation=situation,
            fields=["preferred_option", "reasoning", "purchase_likelihood", "improvement"],
        )
        if extraction is None:
            continue
        raw_opt = str(extraction.get("preferred_option", ""))
        m = re.search(r"[ABC]", raw_opt.upper())
        preferred = m.group(0) if m else None
        raw_lk = extraction.get("purchase_likelihood")
        try:
            likelihood = int(re.search(r"\d", str(raw_lk)).group(0)) if raw_lk is not None else None
        except Exception:
            likelihood = None
        occ = agent.get("occupation")
        occ_title = occ.get("title") if isinstance(occ, dict) else occ
        rows.append({
            "name": agent.get("name"),
            "age": agent.get("age"),
            "gender": agent.get("gender"),
            "occupation": occ_title,
            "preferred_option": preferred,
            "reasoning": extraction.get("reasoning"),
            "purchase_likelihood": likelihood,
            "improvement": extraction.get("improvement"),
        })
    return pd.DataFrame(rows)

results_df = extract_concept_results(
    panel,
    "Concept test for a Gen Z skincare panel. A: Acne Control Serum. B: Barrier Repair Cream. C: Glow Boosting Toner (brightens dull skin and supports a more even complexion).",
)
results_df.to_csv("result.csv", index=False)
display(results_df)

counts = results_df['preferred_option'].value_counts().reindex(['A', 'B', 'C']).fillna(0).astype(int)
print(f"\nConcept counts: A={counts['A']} B={counts['B']} C={counts['C']}")


In [ ]:
control.checkpoint()


## 3. Compare with the claude-persona baseline

Side-by-side check against a parallel run on a different engine (Sonnet, hand-curated 10 personas) — see the [claude-persona genz-skincare demo](https://github.com/takechanman1228/claude-persona/tree/main/demo/genz-skincare) for that run.


In [ ]:
summary = pd.DataFrame({
    'claude-persona (Sonnet, N=10)':   pd.Series({'A': 3, 'B': 4, 'C': 3}),
    'TinyTroupe (gpt-5-mini, N=10)':   counts,
})

print("Concept preference counts:\n")
display(summary)

shares = summary.div(summary.sum(axis=0), axis=1).round(2)
print("\nShare of panel:\n")
display(shares)

fig, ax = plt.subplots(figsize=(9, 5))
summary.T.plot(kind='bar', stacked=True, ax=ax,
               color=['#E4572E', '#669BBC', '#97C879'], edgecolor='black')
ax.set_title('Concept preference: claude-persona vs TinyTroupe')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(title='Concept', labels=['A: Acne Serum', 'B: Barrier Cream', 'C: Glow Toner'],
          bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.show()

mean_pl = results_df['purchase_likelihood'].dropna().mean()
print(f"\nMean purchase likelihood (raw, free-scale, this run): {mean_pl:.2f}")


In [ ]:
control.end()
